# 🏋️‍♂️ Human–Object Interaction (HOI) Multimodal BiLSTM for Sports Action Recognition
## 🚀 Google Colab Cloud-Accelerated Master Notebook (82 Sports Categories)

**Thesis Framework**: *Human–Object Interaction-Aware Multimodal CNN–LSTM Framework for Sports Action Recognition*

### 📋 Overview of this Notebook:
1. **GPU Acceleration**: NVIDIA Tesla T4 / V100 / A100 verification
2. **Codebase Setup**: Automated clone of the GitHub repository and environment install
3. **Dataset Sync**: Mount Google Drive and load `Model Data.zip` (Train 70% / Test 30%)
4. **Multimodal Feature Extraction**: Extract Visual (ResNet-50), Pose (Keypoint R-CNN), Objects (Faster R-CNN), and HOI Kinematics
5. **Model Training**: 2-Layer BiLSTM with Temporal Attention Pooling (Ablations M1–M7)
6. **In-Notebook Visualization**: Live inline display of all 5 thesis outcome diagrams
7. **Persistent Backup**: Auto-saves trained model checkpoints and diagrams directly back to Google Drive

## 1. Verify Colab GPU Acceleration
Make sure your Colab runtime is set to **GPU** (`Runtime -> Change runtime type -> T4 GPU`).

In [ ]:
!nvidia-smi
import torch
print(f"CUDA Ready: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Model  : {torch.cuda.get_device_name(0)}")

## 2. Clone GitHub Codebase & Install Requirements

In [ ]:
# Clone fresh codebase from GitHub
!rm -rf Sport-Action-Detection
!git clone https://github.com/angkon101/Sport-Action-Detection.git
%cd Sport-Action-Detection

# Install dependencies
!pip install -q -r requirements.txt

## 3. Mount Google Drive & Load Dataset (`Model Data`)
Mount your Google Drive to load `Model Data.zip` or `ucf action.zip`.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
drive_path = Path('/content/drive/MyDrive')

# Look for Model Data.zip or ucf action.zip
if (drive_path / 'Model Data.zip').exists():
    print("Found 'Model Data.zip' in Google Drive. Extracting...")
    !unzip -q -o "/content/drive/MyDrive/Model Data.zip" -d ./
elif (drive_path / 'ucf action.zip').exists():
    print("Found 'ucf action.zip' in Google Drive. Extracting and creating Model Data split...")
    !unzip -q -o "/content/drive/MyDrive/ucf action.zip" -d ./
    !python data/scripts/populate_model_data_split.py --source-dir "ucf action" --target-dir "Model Data"
else:
    print("[!] Neither 'Model Data.zip' nor 'ucf action.zip' found in Google Drive root.")
    print("    Please upload your zip file to Google Drive (MyDrive) and rerun this cell.")

## 4. Extract Multimodal Features
Runs ResNet-50, Keypoint R-CNN, Faster R-CNN, and HOI kinematics across 16 sampled frames.

> **Tip**: To run a quick 2-minute smoke test across all 82 categories, append `--max-per-class 2`.

In [ ]:
# Extract multimodal features into data/features
!python run_local.py --extract --data-dir "Model Data" --features-dir "data/features"

# Pre-flight integrity check
!python run_local.py --verify --features-dir "data/features"

## 5. Model Training (BiLSTM + Temporal Attention)
Trains our proposed **M7 Model** on the 70% Train split (6,408 videos) and validates on the 30% Test split (2,743 videos).

In [ ]:
# Train M7 with Cosine Annealing scheduler and AdamW
!python run_local.py --train --features-dir "data/features" --model-mode M7 --epochs 40 --batch-size 32 --lr 0.0005

## 6. View the 5 Publication-Ready Thesis Diagrams

In [ ]:
from IPython.display import Image, display
from pathlib import Path

diag_dir = Path("diagrams/M7")
diagrams = [
    ("Diagram 1: Training & Validation Loss Curves", diag_dir / "loss_curves.png"),
    ("Diagram 2: Top-1 & Top-5 Accuracy Curves", diag_dir / "accuracy_curves.png"),
    ("Diagram 3: 82-Class Confusion Matrix", diag_dir / "confusion_matrix_82.png"),
    ("Diagram 4: Per-Class Accuracy Rankings", diag_dir / "per_class_accuracy.png"),
    ("Diagram 5: Temporal Attention Weights Heatmap", diag_dir / "temporal_attention_weights.png"),
]

for title, img_path in diagrams:
    if img_path.exists():
        print(f"\n{'='*70}\n{title}\n{'='*70}")
        display(Image(filename=str(img_path), width=850))
    else:
        print(f"[!] Diagram not found: {img_path}")

## 7. Backup Checkpoints & Diagrams to Google Drive
Preserves your trained model (`best_model.pth`) and high-resolution diagrams in your Google Drive even after Colab terminates.

In [ ]:
!mkdir -p "/content/drive/MyDrive/Sports_Action_Results"
!cp -r diagrams/ "/content/drive/MyDrive/Sports_Action_Results/"
!cp -r checkpoints/ "/content/drive/MyDrive/Sports_Action_Results/"
!zip -r /content/Sports_Action_Results.zip diagrams checkpoints

print("\n[✓] Results permanently saved to Google Drive: /content/drive/MyDrive/Sports_Action_Results/")
print("[✓] Downloadable archive ready: /content/Sports_Action_Results.zip")

## 8. Real-Time Action Validation & Live Camera Web App
Launches an interactive web app with a public share link (`https://xxxx.gradio.live`).
- Upload any custom sports video (.mp4, .avi, .mov)
- Record live from your webcam directly in the browser
- Real-time visualization of 17 skeletal joints, sports equipment boxes, and HOI vectors
- Live Top-5 prediction confidence chart

In [ ]:
# Launch Gradio Web App with public shareable link
!python app_web.py --share